# DriveValue — Notebook 2: Machine Learning Modeling

**Project:** DriveValue — Nigerian Used Car Price Predictor  
**Target Variable:** `price`  

This notebook covers:
- Data cleaning and feature engineering
- Preprocessing pipeline
- Baseline Linear Regression
- Random Forest model
- Cross validation
- Hyperparameter tuning
- Final evaluation
- Saving the model

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

os.makedirs('../model', exist_ok=True)
os.makedirs('../assets', exist_ok=True)

RANDOM_STATE = 42
print('Libraries loaded!')

## 1. Load Dataset

In [ ]:
path = kagglehub.dataset_download('makindekayode/nigerian-car-prices-dataset')
file_path = os.path.join(path, 'car_prices.csv')
df = pd.read_csv(file_path)

print(f'Dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 2. Cleaning and Feature Engineering

In [ ]:
# Drop duplicates
df = df.drop_duplicates()

# Drop rows where target is missing
df = df.dropna(subset=['price'])

# Feature engineering — car age is more useful than raw year
df['car_age'] = 2025 - df['Year of manufacture']

# Drop columns not useful for modeling
drop_cols = ['car_id', 'car', 'Colour', 'Trim', 'Model']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print(f'Clean dataset shape: {df.shape}')
df.head()

## 3. Define Features and Target

In [ ]:
TARGET = 'price'

NUMERIC_FEATURES = [
    'car_age',
    'Mileage',
    'Engine Size',
    'Horse Power',
    'Number of Cylinders',
    'Seats'
]

CATEGORICAL_FEATURES = [
    'Make',
    'fuel type',
    'gear type',
    'Condition',
    'Drivetrain',
    'Registered city',
    'Selling Condition',
    'Bought Condition'
]

# Keep only available columns
NUMERIC_FEATURES   = [c for c in NUMERIC_FEATURES   if c in df.columns]
CATEGORICAL_FEATURES = [c for c in CATEGORICAL_FEATURES if c in df.columns]
ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

X = df[ALL_FEATURES]
y = df[TARGET]

print(f'Numeric features:     {NUMERIC_FEATURES}')
print(f'Categorical features: {CATEGORICAL_FEATURES}')
print(f'X shape: {X.shape}, y shape: {y.shape}')

# Save feature columns for app
joblib.dump({
    'numeric': NUMERIC_FEATURES,
    'categorical': CATEGORICAL_FEATURES,
    'all': ALL_FEATURES
}, '../model/feature_columns.joblib')
print('Feature columns saved!')

## 4. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f'Train set: {X_train.shape[0]:,} rows')
print(f'Test set:  {X_test.shape[0]:,} rows')

## 5. Build Preprocessing Pipeline

In [ ]:
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, NUMERIC_FEATURES),
    ('cat', categorical_transformer, CATEGORICAL_FEATURES)
])

# Save preprocessor separately
joblib.dump(preprocessor, '../model/preprocessor.joblib')
print('Preprocessor saved!')

## 6. Baseline — Linear Regression

In [ ]:
baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

baseline_pipeline.fit(X_train, y_train)
y_pred_baseline = baseline_pipeline.predict(X_test)

print('=== BASELINE: LINEAR REGRESSION ===')
print('MAE: ', mean_absolute_error(y_test, y_pred_baseline))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_baseline)))
print('R2:  ', r2_score(y_test, y_pred_baseline))

## 7. Random Forest Model

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

rf_pipeline.fit(X_train, y_train)
y_pred_rf = rf_pipeline.predict(X_test)

print('=== RANDOM FOREST ===')
print('MAE: ', mean_absolute_error(y_test, y_pred_rf))
print('RMSE:', np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print('R2:  ', r2_score(y_test, y_pred_rf))

## 8. Cross Validation

In [ ]:
cv_scores = cross_val_score(
    rf_pipeline, X_train, y_train,
    cv=5, scoring='r2', n_jobs=-1
)

print('=== CROSS VALIDATION (5-fold R2) ===')
print(f'Scores: {[round(s, 3) for s in cv_scores]}')
print(f'Mean R2: {cv_scores.mean():.3f}')
print(f'Std R2:  {cv_scores.std():.3f}')

## 9. Hyperparameter Tuning

In [ ]:
param_dist = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4]
}

random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=param_dist,
    n_iter=10,
    cv=3,
    scoring='r2',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print(f'Best params: {random_search.best_params_}')
print(f'Best CV R2:  {random_search.best_score_:.3f}')

## 10. Final Evaluation

In [ ]:
best_model = random_search.best_estimator_
y_pred_final = best_model.predict(X_test)

mae  = mean_absolute_error(y_test, y_pred_final)
rmse = np.sqrt(mean_squared_error(y_test, y_pred_final))
r2   = r2_score(y_test, y_pred_final)

print('=== FINAL MODEL PERFORMANCE ===')
print(f'MAE:  N{mae:,.0f}')
print(f'RMSE: N{rmse:,.0f}')
print(f'R2:   {r2:.3f}')

In [ ]:
# Predicted vs Actual plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred_final, alpha=0.4, color='steelblue')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', label='Perfect prediction')
plt.xlabel('Actual Price (N)')
plt.ylabel('Predicted Price (N)')
plt.title('Predicted vs Actual Car Prices')
plt.legend()
plt.tight_layout()
plt.savefig('../assets/predicted_vs_actual.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature importance plot
rf_model = best_model.named_steps['model']
preprocessor_fitted = best_model.named_steps['preprocessor']

cat_feature_names = preprocessor_fitted.named_transformers_['cat']['encoder'].get_feature_names_out(CATEGORICAL_FEATURES).tolist()
all_feature_names = NUMERIC_FEATURES + cat_feature_names

importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'feature': all_feature_names,
    'importance': importances
}).sort_values('importance', ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x='importance', y='feature', data=feat_imp_df, palette='Blues_r')
plt.title('Top 15 Feature Importances — Random Forest')
plt.tight_layout()
plt.savefig('../assets/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Save Final Model

In [ ]:
joblib.dump(best_model, '../model/car_price_model.joblib')
print('Model saved to ../model/car_price_model.joblib')

# Verify it loads correctly
loaded = joblib.load('../model/car_price_model.joblib')
test_pred = loaded.predict(X_test.head(3))
print(f'Verification — sample predictions: {test_pred.round(0)}')
print('Model verified and ready!')